In [48]:
import pandas as pd

holiday_events = pd.read_csv('data/holidays_events.csv', parse_dates=['date'])
oil = pd.read_csv('data/oil.csv', parse_dates=['date'])
stores = pd.read_csv('data/stores.csv')
transactions = pd.read_csv('data/transactions.csv', parse_dates=['date'])
train_data = pd.read_csv('data/train.csv', parse_dates=['date'])
test_data = pd.read_csv('data/test.csv', parse_dates=['date'])

In [49]:
oil.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        1218 non-null   datetime64[ns]
 1   dcoilwtico  1175 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 19.2 KB


In [50]:
oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()

In [51]:
oil.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        1218 non-null   datetime64[ns]
 1   dcoilwtico  1218 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 19.2 KB


In [52]:
train_data = pd.merge(train_data, stores, on='store_nbr', how='left')
test_data = pd.merge(test_data, stores, on='store_nbr', how='left')

In [53]:
train_data = pd.merge(train_data, oil, on='date', how='left')
test_data = pd.merge(test_data, oil, on='date', how='left')

In [54]:
train_data

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,dcoilwtico
0,0,2013-01-01,1,AUTOMOTIVE,0.000,0,Quito,Pichincha,D,13,93.14
1,1,2013-01-01,1,BABY CARE,0.000,0,Quito,Pichincha,D,13,93.14
2,2,2013-01-01,1,BEAUTY,0.000,0,Quito,Pichincha,D,13,93.14
3,3,2013-01-01,1,BEVERAGES,0.000,0,Quito,Pichincha,D,13,93.14
4,4,2013-01-01,1,BOOKS,0.000,0,Quito,Pichincha,D,13,93.14
...,...,...,...,...,...,...,...,...,...,...,...
3000883,3000883,2017-08-15,9,POULTRY,438.133,0,Quito,Pichincha,B,6,47.57
3000884,3000884,2017-08-15,9,PREPARED FOODS,154.553,1,Quito,Pichincha,B,6,47.57
3000885,3000885,2017-08-15,9,PRODUCE,2419.729,148,Quito,Pichincha,B,6,47.57
3000886,3000886,2017-08-15,9,SCHOOL AND OFFICE SUPPLIES,121.000,8,Quito,Pichincha,B,6,47.57


In [55]:
train_data_idx = len(train_data)

In [56]:
df = pd.concat([train_data, test_data], axis=0, ignore_index=True)
df = df.sort_values(['date', 'store_nbr', 'family']).reset_index(drop=True)

In [57]:
df = pd.merge(df, transactions, on=['date', 'store_nbr'], how='left')

In [58]:
df['transactions'] = df['transactions'].fillna(0)

In [59]:
df['transactions_lag_16'] = df.groupby('store_nbr')['transactions'].transform(lambda x: x.shift(16))
df['transactions_lag_30'] = df.groupby('store_nbr')['transactions'].transform(lambda x: x.shift(30))

In [60]:
df['transactions_roll_mean_7'] = df.groupby('store_nbr')['transactions_lag_16'].transform(lambda x: x.rolling(7).mean())
df['transactions_roll_std_7'] = df.groupby('store_nbr')['transactions_lag_16'].transform(lambda x: x.rolling(7).std())

In [61]:
df['transactions_lag_16'] = df['transactions_lag_16'].fillna(df['transactions_lag_16'].mean())
df['transactions_lag_30'] = df['transactions_lag_30'].fillna(df['transactions_lag_30'].mean())
df['transactions_roll_mean_7'] = df['transactions_roll_mean_7'].fillna(df['transactions_roll_mean_7'].mean())
df['transactions_roll_std_7'] = df['transactions_roll_std_7'].fillna(df['transactions_roll_std_7'].mean())

df['dcoilwtico'] = df['dcoilwtico'].fillna(df['dcoilwtico'].mean())

In [62]:
df['dayofweek'] = df['date'].dt.dayofweek
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['is_salary_day'] = df['day'].isin([15, 16, 30, 31]).astype(int)

In [63]:
df = df.drop(columns=['transactions'])

train_final = df.iloc[:train_data_idx].copy()
test_final = df.iloc[train_data_idx:].copy()

In [64]:
train_final

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,dcoilwtico,transactions_lag_16,transactions_lag_30,transactions_roll_mean_7,transactions_roll_std_7,dayofweek,day,month,is_salary_day
0,0,2013-01-01,1,AUTOMOTIVE,0.000,0,Quito,Pichincha,D,13,93.14,1541.604652,1541.989572,1541.768831,18.043303,1,1,1,0
1,1,2013-01-01,1,BABY CARE,0.000,0,Quito,Pichincha,D,13,93.14,1541.604652,1541.989572,1541.768831,18.043303,1,1,1,0
2,2,2013-01-01,1,BEAUTY,0.000,0,Quito,Pichincha,D,13,93.14,1541.604652,1541.989572,1541.768831,18.043303,1,1,1,0
3,3,2013-01-01,1,BEVERAGES,0.000,0,Quito,Pichincha,D,13,93.14,1541.604652,1541.989572,1541.768831,18.043303,1,1,1,0
4,4,2013-01-01,1,BOOKS,0.000,0,Quito,Pichincha,D,13,93.14,1541.604652,1541.989572,1541.768831,18.043303,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3000883,3000751,2017-08-15,54,POULTRY,59.619,0,El Carmen,Manabi,C,3,47.57,802.000000,818.000000,802.000000,0.000000,1,15,8,1
3000884,3000752,2017-08-15,54,PREPARED FOODS,94.000,0,El Carmen,Manabi,C,3,47.57,802.000000,818.000000,802.000000,0.000000,1,15,8,1
3000885,3000753,2017-08-15,54,PRODUCE,915.371,76,El Carmen,Manabi,C,3,47.57,802.000000,802.000000,802.000000,0.000000,1,15,8,1
3000886,3000754,2017-08-15,54,SCHOOL AND OFFICE SUPPLIES,0.000,0,El Carmen,Manabi,C,3,47.57,802.000000,802.000000,802.000000,0.000000,1,15,8,1


In [65]:
# import numpy as np
#
# train_final['target_log'] = np.log1p(train_final['sales'])

In [66]:
train_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 19 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   id                        int64         
 1   date                      datetime64[ns]
 2   store_nbr                 int64         
 3   family                    object        
 4   sales                     float64       
 5   onpromotion               int64         
 6   city                      object        
 7   state                     object        
 8   type                      object        
 9   cluster                   int64         
 10  dcoilwtico                float64       
 11  transactions_lag_16       float64       
 12  transactions_lag_30       float64       
 13  transactions_roll_mean_7  float64       
 14  transactions_roll_std_7   float64       
 15  dayofweek                 int32         
 16  day                       int32         
 17  month   

In [67]:
start_date = '2016-01-01'
train_final = train_final[train_final['date'] >= start_date]

In [68]:
split_date = '2017-07-31'
train_data = train_final[train_final['date'] < split_date]
val_data = train_final[train_final['date'] >= split_date]

In [69]:
features = [
    'store_nbr', 'onpromotion', 'cluster', 'dcoilwtico',
    'transactions_lag_16', 'transactions_lag_30',
    'transactions_roll_mean_7', 'transactions_roll_std_7',
    'dayofweek', 'day', 'month', 'is_salary_day', 'family',
    'city', 'state', 'type'
]

X_train = train_data[features]
y_train = train_data['sales']

X_val = val_data[features]
y_val = val_data['sales']

In [70]:
cat_features = ['family', 'city', 'state', 'type']

In [71]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.03,
    depth=5,
    eval_metric='MAPE',
    early_stopping_rounds=100,
    random_seed=42
)

model.fit(
    X_train, y_train,
    # eval_set=[(X_val, y_val)],
    cat_features=cat_features,
    # verbose=100
)

0:	learn: 122.8562573	total: 637ms	remaining: 15m 55s
1:	learn: 120.1062339	total: 1.63s	remaining: 20m 19s
2:	learn: 117.4452523	total: 2.42s	remaining: 20m 8s
3:	learn: 114.9293271	total: 3.31s	remaining: 20m 38s
4:	learn: 112.4960575	total: 4.02s	remaining: 20m 3s
5:	learn: 110.0456508	total: 4.72s	remaining: 19m 34s
6:	learn: 107.6653130	total: 5.57s	remaining: 19m 47s
7:	learn: 105.3699180	total: 6.24s	remaining: 19m 22s
8:	learn: 103.4276398	total: 6.78s	remaining: 18m 43s
9:	learn: 101.4464771	total: 7.51s	remaining: 18m 39s
10:	learn: 99.4799059	total: 8.14s	remaining: 18m 22s
11:	learn: 97.4920540	total: 8.72s	remaining: 18m 1s
12:	learn: 95.3988018	total: 9.47s	remaining: 18m 3s
13:	learn: 93.2198620	total: 10.1s	remaining: 17m 56s
14:	learn: 91.4919300	total: 10.8s	remaining: 17m 47s
15:	learn: 88.9641444	total: 11.4s	remaining: 17m 41s
16:	learn: 86.5561750	total: 12.1s	remaining: 17m 34s
17:	learn: 84.2166853	total: 12.7s	remaining: 17m 26s
18:	learn: 81.9584316	total: 13.

CatBoostRegressor(depth=5, early_stopping_rounds=100, eval_metric='MAPE', iterations=1500, learning_rate=0.03, loss_function='RMSE', random_seed=42)

In [74]:
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error

y_pred = model.predict(X_val)
y_pred = np.clip(y_pred, 0, None)

mae = mean_absolute_error(y_val, y_pred)
# mape = mean_absolute_percentage_error(y_val, y_pred)

print(f'В среднем отклонение на {mae:.2f} единицы выручки')
wmape = np.sum(np.abs(y_val - y_pred)) / np.sum(y_val)
print(f'WMAPE: {wmape * 100:.2f}%')

В среднем отклонение на 102.76 единицы выручки
WMAPE: 22.00%
